# Data Rebalancing con approccio Ibrido (RandomUnderSampler + SMOTE) - v4 Output

In questo notebook, applichiamo una tecnica di rebalancing ibrida per gestire il dataset sbilanciato. L'approccio combina:

1.  **RandomUnderSampler**: Per ridurre la classe maggioritaria (`BenignPositive`).
2.  **SMOTE (Synthetic Minority Over-sampling Technique)**: Per aumentare le classi minoritarie (`FalsePositive` e `TruePositive`) generando campioni sintetici.

**Aggiornamento v4:**
- Input: `processed_v4_noleakage` (Training set pulito senza leakage)
- Output: `processed_v4_hybrid`


In [ ]:
import pandas as pd
import numpy as np
import os
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from collections import Counter

# Definisci e crea la directory di output per il nuovo dataset
output_dir = '../data/processed_v4_hybrid/'
os.makedirs(output_dir, exist_ok=True)

print(f"Directory di output creata: {output_dir}")

In [ ]:
# Caricamento dei dati di training e test (v4 no leakage)
X_train = pd.read_csv('../data/processed_v4_noleakage/X_train.csv')
y_train = pd.read_csv('../data/processed_v4_noleakage/y_train.csv').squeeze()
X_test = pd.read_csv('../data/processed_v4_noleakage/X_test.csv')
y_test = pd.read_csv('../data/processed_v4_noleakage/y_test.csv').squeeze()

print("Dimensioni dei dati caricati (v4):")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

print("\n" + "="*50)
print("DISTRIBUZIONE CLASSI ORIGINALE (TRAINING)")
print("="*50)
class_counts_original = pd.Series(y_train).value_counts().sort_index()
for cls, count in class_counts_original.items():
    percentage = count / len(y_train) * 100
    print(f"Classe {cls}: {count:,} ({percentage:.2f}%)")
print(f"Total: {len(y_train):,}")

In [ ]:
print("\n" + "="*50)
print("STEP 1: RandomUnderSampler")
print("="*50)

# Calcola le classi attuali
n_majority = (y_train == 0).sum()  # Classe 0 (Non-TP)
n_minority = (y_train == 1).sum()  # Classe 1 (TP)

print(f"Prima del resampling:")
print(f"  Classe 0 (Non-TP): {n_majority:,}")
print(f"  Classe 1 (TP):     {n_minority:,}")
print(f"  Ratio: {n_majority/n_minority:.2f}:1")

# Target: ridurre la classe maggioritaria a 1.5x la classe minoritaria
n_majority_target = int(n_minority * 1.5)

sampling_strategy = {0: n_majority_target}  # Riduci classe 0

print(f"\nTarget dopo RandomUnderSampler:")
print(f"  Classe 0 (Non-TP): {n_majority_target:,}")
print(f"  Classe 1 (TP):     {n_minority:,}")
print(f"  Ratio target: 1.5:1")

rus = RandomUnderSampler(
    sampling_strategy=sampling_strategy,
    random_state=42
)

X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)

print(f"\nDopo RandomUnderSampler:")
class_counts_rus = pd.Series(y_train_resampled).value_counts().sort_index()
for cls, count in class_counts_rus.items():
    percentage = count / len(y_train_resampled) * 100
    print(f"  Classe {cls}: {count:,} ({percentage:.2f}%)")
print(f"  Total: {len(y_train_resampled):,}")


In [ ]:
print("\n" + "="*50)
print("STEP 2: SMOTE")
print("="*50)

# Ora bilancia le classi usando SMOTE per portare la classe minoritaria (1) 
# allo stesso livello della classe maggioritaria (0)
n_majority_after_rus = (y_train_resampled == 0).sum()

# SMOTE per bilanciare completamente: porta classe 1 allo stesso numero di classe 0
sampling_strategy_smote = {1: n_majority_after_rus}

print(f"Target SMOTE:")
print(f"  Classe 0 (Non-TP): {n_majority_after_rus:,} (rimane invariata)")
print(f"  Classe 1 (TP):     {n_majority_after_rus:,} (oversampling con SMOTE)")
print(f"  Ratio target: 1:1 (bilanciamento perfetto)")

smote = SMOTE(
    sampling_strategy=sampling_strategy_smote,
    random_state=42,
    k_neighbors=5
)

X_train_balanced, y_train_balanced = smote.fit_resample(X_train_resampled, y_train_resampled)

print(f"\n" + "="*50)
print("RISULTATO FINALE (dopo RandomUnderSampler + SMOTE)")
print("="*50)
print(f"X_train_balanced: {X_train_balanced.shape}")
print(f"y_train_balanced: {y_train_balanced.shape}")
print("\nDistribuzione finale delle classi:")
class_counts_final = pd.Series(y_train_balanced).value_counts().sort_index()
for cls, count in class_counts_final.items():
    percentage = count / len(y_train_balanced) * 100
    label = "Non-TP" if cls == 0 else "TP"
    print(f"  Classe {cls} ({label}): {count:,} ({percentage:.2f}%)")
print(f"  Total: {len(y_train_balanced):,}")

# Calcola quanti campioni sintetici sono stati generati
n_synthetic = class_counts_final[1] - class_counts_rus[1]
print(f"\n✅ Generati {n_synthetic:,} campioni sintetici per la classe minoritaria (TP)")


In [ ]:
# %%
# ==============================================================================
# MODIFICA PER TEST: USARE SOLO I DATI DELL'UNDERSAMPLER
# ==============================================================================
# Per testare l'efficacia del solo RandomUnderSampler, sovrascriviamo i
# dataframe finali con quelli generati al primo step (prima di SMOTE).
# In questo modo, il resto dello script salverà questi dati.
# POV: non è efficace tanto come l'approccio ibrido, ma utile per testare.
# ==============================================================================

#print("⚠️ MODIFICA ATTIVA: Verranno salvati solo i dati dell'undersampling (pre-SMOTE).")

#X_train_balanced = X_train_resampled.copy()
#y_train_balanced = y_train_resampled.copy()

#print("\nNuove dimensioni per il salvataggio:")
#print(f"X_train_balanced: {X_train_balanced.shape}")
#print(f"y_train_balanced: {y_train_balanced.shape}")


import seaborn as sns
import matplotlib.pyplot as plt

print("\n" + "=" * 50)
print("MATRICE DI CORRELAZIONE")
print("=" * 50)

# Calcola la matrice di correlazione
correlation_matrix = X_train_balanced.corr()

# Imposta la dimensione della figura per una migliore leggibilità
plt.figure(figsize=(20, 18))

# Crea la heatmap utilizzando seaborn
sns.heatmap(correlation_matrix, cmap='coolwarm', annot=False)  # annot=False perché le feature sono troppe

# Aggiungi un titolo
plt.title('Matrice di Correlazione delle Feature su X_train_balanced', fontsize=16)

# Mostra il plot
plt.tight_layout()
plt.show()




In [ ]:
# Salvataggio dei nuovi dati
print("\n" + "="*50)
print("SALVATAGGIO DATI")
print("="*50)
print(f"Directory output: {output_dir}")

# Salva i dati di training ribilanciati
X_train_balanced.to_csv(os.path.join(output_dir, 'X_train.csv'), index=False)
y_train_balanced_df = pd.DataFrame(y_train_balanced, columns=['BinaryIncidentGrade'])
y_train_balanced_df.to_csv(os.path.join(output_dir, 'y_train.csv'), index=False)

# Copia i dati di test originali nella nuova cartella (NON modificare il test set!)
X_test.to_csv(os.path.join(output_dir, 'X_test.csv'), index=False)
y_test_df = pd.DataFrame(y_test, columns=['BinaryIncidentGrade'])
y_test_df.to_csv(os.path.join(output_dir, 'y_test.csv'), index=False)

print("\n✅ Salvataggio completato!")
print(f"\nFile creati:")
print(f"  - X_train.csv: {X_train_balanced.shape}")
print(f"  - y_train.csv: {y_train_balanced.shape}")
print(f"  - X_test.csv: {X_test.shape} (invariato)")
print(f"  - y_test.csv: {y_test.shape} (invariato)")

print(f"\n⚠️  IMPORTANTE: Il test set NON è stato modificato!")
print(f"Solo il training set è stato bilanciato con RandomUnderSampler + SMOTE")
